# Examen 1 - Reporte Experimental Final

Notebook orientado al reporte IEEE para:
- inspeccionar y visualizar el conjunto de datos;
- entrenar el **MLP principal** con la arquitectura base del examen;
- comparar contra **Random Forest** como modelo adicional sustentado en literatura;
- generar tablas, métricas, matrices de confusión e importancia de variables.

**Escenario central:** clasificación de `GradeClass` usando el dataset completo, incluyendo `GPA`.

## 1. Librerías y configuración

La semilla se fija para hacer reproducibles, en la medida de lo posible, las corridas.

In [ ]:
from __future__ import annotations

import os
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score, precision_score, recall_score
from sklearn.model_selection import ParameterGrid, train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from tensorflow import keras

SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

CANDIDATE_DIRS = [
    Path.cwd(),
    Path('/home/dgx/UABC/RRNN/desercion_escolar'),
    Path('/Volumes/dgx/UABC/RRNN/desercion_escolar'),
]
BASE_DIR = next((p for p in CANDIDATE_DIRS if (p / 'Student_performance_data_clean.csv').exists()), Path.cwd())
DATASET = BASE_DIR / 'Student_performance_data_clean.csv'
TARGET = 'GradeClass'
CONTINUOUS = ['Age', 'StudyTimeWeekly', 'Absences', 'GPA']
CATEGORICAL = ['Gender', 'Ethnicity', 'ParentalEducation', 'Tutoring', 'ParentalSupport', 'Extracurricular', 'Sports', 'Music', 'Volunteering']
DISPLAY_NAMES = {
    'StudentID': 'Matrícula',
    'Age': 'edad',
    'StudyTimeWeekly': 'tiempo de estudio semanal',
    'Absences': 'ausencias',
    'GPA': 'promedio general (GPA)',
    'GradeClass': 'clase de calificación',
    'Gender': 'género',
    'Ethnicity': 'grupo étnico',
    'ParentalEducation': 'educación parental',
    'Tutoring': 'tutorías',
    'ParentalSupport': 'apoyo parental',
    'Extracurricular': 'actividades extracurriculares',
    'Sports': 'deportes',
    'Music': 'música',
    'Volunteering': 'voluntariado',
}

MLP_GRID = {
    'optimizer': ['adam', 'rmsprop', 'sgd'],
    'learning_rate': [1e-4, 5e-4, 1e-3],
    'batch_size': [32, 64, 128],
    'epochs': [50, 100, 150],
}

RF_GRID = {
    'n_estimators': [200, 400, 700],
    'max_depth': [None, 10, 20, 30],
    'min_samples_leaf': [1, 2, 4],
    'class_weight': [None, 'balanced', 'balanced_subsample'],
}

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 160)
sns.set_theme(style='whitegrid', palette='deep')
print('BASE_DIR =', BASE_DIR)
print('DATASET =', DATASET)
print('Existe dataset =', DATASET.exists())

## 2. Carga y revisión básica del conjunto de datos

In [ ]:
df = pd.read_csv(DATASET)
df[TARGET] = df[TARGET].astype(int)
df['StudyTimeWeekly'] = df['StudyTimeWeekly'].astype(float)
df['GPA'] = df['GPA'].astype(float)

columnas_mostrar = {k: (v.capitalize() if k != 'StudentID' else v) for k, v in DISPLAY_NAMES.items()}
print('Forma del dataset:', df.shape)
display(df.head().rename(columns=columnas_mostrar))

print('Tipos de datos:')
display(df.dtypes.rename(index=columnas_mostrar).to_frame('tipo'))

print('Valores nulos por columna:')
display(df.isna().sum().rename(index=columnas_mostrar).to_frame('nulos'))

print('Distribución de clases:')
dist_clases = df[TARGET].value_counts().sort_index().to_frame('cantidad')
dist_clases.index.name = 'Clase'
display(dist_clases)

## 3. Visualización exploratoria

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8, 4))
sns.countplot(data=df, x=TARGET, hue=TARGET, dodge=False, palette='viridis', legend=False, ax=ax)
ax.set_title('Distribución de clases')
ax.set_xlabel('Clase')
ax.set_ylabel('Cantidad de estudiantes')
plt.tight_layout()
plt.savefig(BASE_DIR / 'grafica_distribucion_clases.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, col in zip(axes.flat, CONTINUOUS):
    sns.histplot(df[col], bins=20, kde=True, ax=ax, color='#2E86AB')
    ax.set_title(f"Distribución de {DISPLAY_NAMES[col]}")
    ax.set_xlabel(DISPLAY_NAMES[col])
    ax.set_ylabel('Frecuencia')
plt.tight_layout()
plt.savefig(BASE_DIR / 'graficas_distribucion_variables_continuas.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
numeric_cols = ['Age', 'Gender', 'Ethnicity', 'ParentalEducation', 'StudyTimeWeekly', 'Absences', 'Tutoring', 'ParentalSupport', 'Extracurricular', 'Sports', 'Music', 'Volunteering', 'GPA', 'GradeClass']
corr = df[numeric_cols].corr(numeric_only=True)
corr = corr.rename(index=columnas_mostrar, columns=columnas_mostrar)
plt.figure(figsize=(11, 8))
sns.heatmap(corr, cmap='coolwarm', center=0, annot=False)
plt.title('Matriz de correlación')
plt.tight_layout()
plt.savefig(BASE_DIR / 'matriz_correlacion.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, col in zip(axes.flat, CONTINUOUS):
    sns.boxplot(data=df, x=TARGET, y=col, hue=TARGET, dodge=False, legend=False, ax=ax, palette='Set2')
    ax.set_title(f"{DISPLAY_NAMES[col].capitalize()} por clase")
    ax.set_xlabel('Clase')
    ax.set_ylabel(DISPLAY_NAMES[col])
plt.tight_layout()
plt.savefig(BASE_DIR / 'boxplots_variables_por_clase.png', dpi=200, bbox_inches='tight')
plt.show()

## 4. Preparación de datos

- Se elimina implícitamente `StudentID` por ser un identificador.
- Se conserva `GPA` porque el enunciado no prohíbe su uso y la explicación oral del profesor sugiere que el dataset debe permitir desempeños superiores al 80%.
- Se usa una **partición estratificada 70/15/15**.

In [ ]:
feature_columns = CONTINUOUS + CATEGORICAL
X_df = df[feature_columns].copy()
y = df[TARGET].to_numpy(dtype=np.int32)

X_train_df, X_temp_df, y_train, y_temp = train_test_split(
    X_df, y, test_size=0.30, stratify=y, random_state=SEED
)
X_val_df, X_test_df, y_val, y_test = train_test_split(
    X_temp_df, y_temp, test_size=0.50, stratify=y_temp, random_state=SEED
)

preprocessor = ColumnTransformer([
    ('continuous', StandardScaler(), CONTINUOUS),
    ('categorical', OneHotEncoder(handle_unknown='ignore', sparse_output=False), CATEGORICAL),
])

X_train_mlp = np.asarray(preprocessor.fit_transform(X_train_df), dtype=np.float32)
X_val_mlp = np.asarray(preprocessor.transform(X_val_df), dtype=np.float32)
X_test_mlp = np.asarray(preprocessor.transform(X_test_df), dtype=np.float32)

X_train_rf = X_train_df.astype(float).copy()
X_val_rf = X_val_df.astype(float).copy()
X_test_rf = X_test_df.astype(float).copy()

print('Train MLP:', X_train_mlp.shape, 'Val MLP:', X_val_mlp.shape, 'Test MLP:', X_test_mlp.shape)
print('Train RF:', X_train_rf.shape, 'Val RF:', X_val_rf.shape, 'Test RF:', X_test_rf.shape)
print('Clases en entrenamiento:', np.unique(y_train))

## 5. Funciones auxiliares

In [ ]:
def set_seed(seed: int = SEED) -> None:
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

def collect_metrics(y_true, y_pred):
    return {
        'accuracy': accuracy_score(y_true, y_pred),
        'precision_macro': precision_score(y_true, y_pred, average='macro', zero_division=0),
        'recall_macro': recall_score(y_true, y_pred, average='macro', zero_division=0),
        'f1_macro': f1_score(y_true, y_pred, average='macro', zero_division=0),
    }

def build_optimizer(name: str, learning_rate: float):
    if name == 'adam':
        return keras.optimizers.Adam(learning_rate=learning_rate)
    if name == 'rmsprop':
        return keras.optimizers.RMSprop(learning_rate=learning_rate)
    return keras.optimizers.SGD(learning_rate=learning_rate, momentum=0.9, nesterov=True)

def build_exam_mlp(input_dim: int, num_classes: int, optimizer_name: str, learning_rate: float):
    model = keras.Sequential([
        keras.layers.Input(shape=(input_dim,)),
        keras.layers.Dense(200, activation='relu'),
        keras.layers.Dropout(0.2),
        keras.layers.Dense(200, activation='relu'),
        keras.layers.Dropout(0.2),
        keras.layers.Dense(150, activation='relu'),
        keras.layers.Dropout(0.2),
        keras.layers.Dense(200, activation='relu'),
        keras.layers.Dropout(0.2),
        keras.layers.Dense(len(np.unique(y_train)), activation='softmax'),
    ])
    model.compile(
        optimizer=build_optimizer(optimizer_name, learning_rate),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy'],
    )
    return model

## 6. Búsqueda de hiperparámetros para el MLP principal

In [ ]:
best_mlp_model = None
best_mlp_cfg = None
best_mlp_history = None
best_mlp_score = -np.inf
best_mlp_loss = np.inf
mlp_rows = []

for cfg in ParameterGrid(MLP_GRID):
    set_seed()
    model = build_exam_mlp(X_train_mlp.shape[1], len(np.unique(y_train)), cfg['optimizer'], cfg['learning_rate'])
    callbacks = [keras.callbacks.EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)]
    history = model.fit(
        X_train_mlp, y_train,
        validation_data=(X_val_mlp, y_val),
        epochs=cfg['epochs'],
        batch_size=cfg['batch_size'],
        verbose=0,
        callbacks=callbacks,
    )
    pred_val = np.argmax(model.predict(X_val_mlp, verbose=0), axis=1)
    metrics_val = collect_metrics(y_val, pred_val)
    val_loss = float(np.min(history.history['val_loss']))
    row = {
        **cfg,
        'epochs_ran': len(history.history['loss']),
        'val_accuracy': round(metrics_val['accuracy'], 4),
        'val_f1_macro': round(metrics_val['f1_macro'], 4),
        'val_recall_macro': round(metrics_val['recall_macro'], 4),
        'val_loss': round(val_loss, 4),
    }
    mlp_rows.append(row)
    if metrics_val['f1_macro'] > best_mlp_score or (np.isclose(metrics_val['f1_macro'], best_mlp_score) and val_loss < best_mlp_loss):
        best_mlp_score = metrics_val['f1_macro']
        best_mlp_loss = val_loss
        best_mlp_model = model
        best_mlp_cfg = dict(cfg)
        best_mlp_history = history.history

mlp_search_df = pd.DataFrame(mlp_rows).sort_values(['val_f1_macro', 'val_accuracy'], ascending=[False, False])
print('Mejor configuración MLP:', best_mlp_cfg)
display(mlp_search_df.head(12))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(best_mlp_history['loss'], label='Entrenamiento')
axes[0].plot(best_mlp_history['val_loss'], label='Validación')
axes[0].set_title('Pérdida del mejor MLP')
axes[0].set_xlabel('Época')
axes[0].legend()

axes[1].plot(best_mlp_history['accuracy'], label='Entrenamiento')
axes[1].plot(best_mlp_history['val_accuracy'], label='Validación')
axes[1].set_title('Exactitud del mejor MLP')
axes[1].set_xlabel('Época')
axes[1].legend()
plt.tight_layout()
plt.savefig(BASE_DIR / 'curvas_entrenamiento_mlp.png', dpi=200, bbox_inches='tight')
plt.show()

## 6.1 Diagrama de la arquitectura del MLP principal

In [ ]:
def draw_network_mlp_examen(input_dim: int, num_classes: int):
    fig, ax = plt.subplots(figsize=(15, 7), facecolor='#f7f7f7')
    ax.set_facecolor('#f7f7f7')
    ax.axis('off')

    xp = {'input': 0, 'dense1': 2, 'dense2': 4, 'dense3': 6, 'dense4': 8, 'output': 10}
    colors = {
        'input': '#6C5CE7',
        'dense1': '#E84393',
        'dense2': '#F39C12',
        'dense3': '#2ECC71',
        'dense4': '#E74C3C',
        'output': '#D4A017',
    }
    labels = {
        'input': f'Entrada\n({input_dim} variables)',
        'dense1': 'Densa + ReLU\nDropout(0.2)',
        'dense2': 'Densa + ReLU\nDropout(0.2)',
        'dense3': 'Densa + ReLU\nDropout(0.2)',
        'dense4': 'Densa + ReLU\nDropout(0.2)',
        'output': f'Salida\n{num_classes}, Softmax',
    }
    sizes = {'input': str(input_dim), 'dense1': '200', 'dense2': '200', 'dense3': '150', 'dense4': '200', 'output': str(num_classes)}
    y_nodes = [4.5, 3.5, 2.5, 1.5, 0.5]

    def draw_column(x, color):
        for y in y_nodes:
            circ = plt.Circle((x, y), 0.18, color=color, ec='white', lw=2, alpha=0.95)
            ax.add_patch(circ)
        ax.text(x, 2.0, '⋯', ha='center', va='center', fontsize=18, color='#777777')

    def connect_columns(x1, x2):
        for y1 in y_nodes:
            for y2 in y_nodes:
                ax.plot([x1 + 0.18, x2 - 0.18], [y1, y2], color='#cccccc', lw=0.8, alpha=0.35, zorder=0)

    order = ['input', 'dense1', 'dense2', 'dense3', 'dense4', 'output']
    for left, right in zip(order[:-1], order[1:]):
        connect_columns(xp[left], xp[right])
    for key in order:
        draw_column(xp[key], colors[key])
        ax.text(
            xp[key], 5.5, labels[key],
            ha='center', va='center', fontsize=10, fontweight='bold', color='white',
            bbox=dict(boxstyle='round,pad=0.35', facecolor=colors[key], edgecolor='none', alpha=0.95)
        )
        ax.text(
            xp[key], -0.1, f'n = {sizes[key]}',
            ha='center', fontsize=8, color='#555555',
            bbox=dict(boxstyle='round,pad=0.25', facecolor='white', edgecolor=colors[key], alpha=0.9)
        )

    edge_labels = [('input', 'dense1', '$W_1, b_1$'), ('dense1', 'dense2', '$W_2, b_2$'), ('dense2', 'dense3', '$W_3, b_3$'), ('dense3', 'dense4', '$W_4, b_4$'), ('dense4', 'output', '$W_5, b_5$')]
    for left, right, txt in edge_labels:
        xm = (xp[left] + xp[right]) / 2
        ax.text(xm, 4.95, txt, ha='center', va='bottom', fontsize=9, color='#666666')

    ax.set_title(
        f'Arquitectura del MLP principal del examen ({input_dim} → 200 → 200 → 150 → 200 → {num_classes})',
        fontsize=14, fontweight='bold', pad=14, color='#222222'
    )
    ax.set_xlim(-0.7, 10.7)
    ax.set_ylim(-0.5, 6.1)
    plt.tight_layout()
    plt.savefig(BASE_DIR / 'arquitectura_mlp_principal.png', dpi=160, bbox_inches='tight', facecolor=fig.get_facecolor())
    plt.show()
    print('Diagrama guardado en:', BASE_DIR / 'arquitectura_mlp_principal.png')

draw_network_mlp_examen(X_train_mlp.shape[1], len(np.unique(y_train)))

In [ ]:
plt.figure(figsize=(10, 4))
top_mlp = mlp_search_df.head(10).copy()
top_mlp = top_mlp.rename(columns={
    'optimizer': 'optimizador',
    'learning_rate': 'tasa_aprendizaje',
    'batch_size': 'tamano_lote',
    'epochs': 'epocas_solicitadas',
    'epochs_ran': 'epocas_ejecutadas',
    'val_accuracy': 'exactitud_validacion',
    'val_f1_macro': 'puntaje_f1_macro_validacion',
    'val_recall_macro': 'sensibilidad_macro_validacion',
    'val_loss': 'perdida_validacion',
})
top_mlp['configuracion'] = top_mlp['optimizador'] + ' | tasa=' + top_mlp['tasa_aprendizaje'].astype(str) + ' | lote=' + top_mlp['tamano_lote'].astype(str)
sns.barplot(data=top_mlp, x='puntaje_f1_macro_validacion', y='configuracion', color='#4C78A8')
plt.title('Mejores configuraciones del MLP según puntaje F1 macro en validación')
plt.xlabel('Puntaje F1 macro')
plt.ylabel('Configuración')
plt.tight_layout()
plt.savefig(BASE_DIR / 'comparacion_hiperparametros_mlp.png', dpi=200, bbox_inches='tight')
plt.show()

## 7. Búsqueda de hiperparámetros para Random Forest

In [ ]:
best_rf_model = None
best_rf_cfg = None
best_rf_score = -np.inf
rf_rows = []

for cfg in ParameterGrid(RF_GRID):
    model = RandomForestClassifier(random_state=SEED, n_jobs=-1, **cfg)
    model.fit(X_train_rf, y_train)
    pred_val = model.predict(X_val_rf)
    metrics_val = collect_metrics(y_val, pred_val)
    row = {
        **cfg,
        'val_accuracy': round(metrics_val['accuracy'], 4),
        'val_f1_macro': round(metrics_val['f1_macro'], 4),
        'val_recall_macro': round(metrics_val['recall_macro'], 4),
    }
    rf_rows.append(row)
    if metrics_val['f1_macro'] > best_rf_score:
        best_rf_score = metrics_val['f1_macro']
        best_rf_model = model
        best_rf_cfg = dict(cfg)

rf_search_df = pd.DataFrame(rf_rows).sort_values(['val_f1_macro', 'val_accuracy'], ascending=[False, False])
print('Mejor configuración Random Forest:', best_rf_cfg)
display(rf_search_df.head(12))

In [ ]:
plt.figure(figsize=(10, 4))
top_rf = rf_search_df.head(10).copy()
top_rf = top_rf.rename(columns={
    'n_estimators': 'numero_arboles',
    'max_depth': 'profundidad_maxima',
    'min_samples_leaf': 'min_muestras_hoja',
    'class_weight': 'peso_clase',
    'val_accuracy': 'exactitud_validacion',
    'val_f1_macro': 'puntaje_f1_macro_validacion',
    'val_recall_macro': 'sensibilidad_macro_validacion',
})
top_rf['configuracion'] = (
    'árboles=' + top_rf['numero_arboles'].astype(str)
    + ' | prof=' + top_rf['profundidad_maxima'].astype(str)
    + ' | hoja=' + top_rf['min_muestras_hoja'].astype(str)
)
sns.barplot(data=top_rf, x='puntaje_f1_macro_validacion', y='configuracion', color='#59A14F')
plt.title('Mejores configuraciones de Random Forest según puntaje F1 macro en validación')
plt.xlabel('Puntaje F1 macro')
plt.ylabel('Configuración')
plt.tight_layout()
plt.savefig(BASE_DIR / 'comparacion_hiperparametros_random_forest.png', dpi=200, bbox_inches='tight')
plt.show()

## 8. Evaluación final en el conjunto de prueba

In [ ]:
labels = sorted(np.unique(y_test).tolist())
mlp_pred = np.argmax(best_mlp_model.predict(X_test_mlp, verbose=0), axis=1)
rf_pred = best_rf_model.predict(X_test_rf)

mlp_metrics = collect_metrics(y_test, mlp_pred)
rf_metrics = collect_metrics(y_test, rf_pred)

comparison_df = pd.DataFrame([
    {'modelo': 'MLP principal', 'mejor_configuracion': str(best_mlp_cfg), **{k: round(v, 4) for k, v in mlp_metrics.items()}},
    {'modelo': 'Random Forest', 'mejor_configuracion': str(best_rf_cfg), **{k: round(v, 4) for k, v in rf_metrics.items()}},
])
comparison_df = comparison_df.rename(columns={
    'accuracy': 'exactitud',
    'precision_macro': 'precisión_macro',
    'recall_macro': 'sensibilidad_macro',
    'f1_macro': 'puntaje_f1_macro',
})
display(comparison_df)

In [ ]:
report_mlp_df = pd.DataFrame(classification_report(y_test, mlp_pred, output_dict=True, zero_division=0)).transpose().reset_index().rename(columns={'index': 'clase'})
report_rf_df = pd.DataFrame(classification_report(y_test, rf_pred, output_dict=True, zero_division=0)).transpose().reset_index().rename(columns={'index': 'clase'})

print('Reporte de clasificación - MLP principal')
display(report_mlp_df)

print('Reporte de clasificación - Random Forest')
display(report_rf_df)

## 9. Matrices de confusión normalizadas

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
cm_mlp = confusion_matrix(y_test, mlp_pred, labels=labels, normalize='true')
cm_rf = confusion_matrix(y_test, rf_pred, labels=labels, normalize='true')
sns.heatmap(cm_mlp, annot=True, fmt='.2f', cmap='Blues', xticklabels=labels, yticklabels=labels, ax=axes[0])
axes[0].set_title('MLP principal')
axes[0].set_xlabel('Clase predicha')
axes[0].set_ylabel('Clase real')
sns.heatmap(cm_rf, annot=True, fmt='.2f', cmap='Greens', xticklabels=labels, yticklabels=labels, ax=axes[1])
axes[1].set_title('Random Forest')
axes[1].set_xlabel('Clase predicha')
axes[1].set_ylabel('Clase real')
plt.tight_layout()
plt.savefig(BASE_DIR / 'matrices_confusion_comparadas.png', dpi=200, bbox_inches='tight')
plt.show()

## 10. Comparación visual de métricas

In [ ]:
plot_df = comparison_df.melt(id_vars=['modelo', 'mejor_configuracion'], value_vars=['exactitud', 'precisión_macro', 'sensibilidad_macro', 'puntaje_f1_macro'], var_name='metrica', value_name='valor')
plot_df['metrica'] = plot_df['metrica'].replace({
    'exactitud': 'Exactitud',
    'precisión_macro': 'Precisión macro',
    'sensibilidad_macro': 'Sensibilidad macro',
    'puntaje_f1_macro': 'Puntaje F1 macro',
})
plt.figure(figsize=(10, 5))
sns.barplot(data=plot_df, x='metrica', y='valor', hue='modelo')
plt.ylim(0, 1.0)
plt.title('Comparación de métricas en prueba')
plt.xlabel('Métrica')
plt.ylabel('Valor')
plt.tight_layout()
plt.savefig(BASE_DIR / 'comparacion_metricas_modelos.png', dpi=200, bbox_inches='tight')
plt.show()

## 11. Importancia de variables con Random Forest

In [ ]:
importance_df = pd.DataFrame({'variable': X_train_rf.columns, 'importancia': best_rf_model.feature_importances_}).sort_values('importancia', ascending=False).reset_index(drop=True)
display(importance_df.head(15))

plt.figure(figsize=(10, 5))
top_importance = importance_df.head(12)
sns.barplot(data=top_importance, x='importancia', y='variable', hue='variable', dodge=False, palette='viridis', legend=False)
plt.title('Variables más relevantes según Random Forest')
plt.xlabel('Importancia')
plt.ylabel('Variable')
plt.tight_layout()
plt.savefig(BASE_DIR / 'importancia_variables_random_forest.png', dpi=200, bbox_inches='tight')
plt.show()

## 12. Exportación de artefactos para el reporte

In [ ]:
comparison_df.to_csv(BASE_DIR / 'comparacion_modelos_final.csv', index=False)
mlp_search_df.to_csv(BASE_DIR / 'hiperparametros_mlp_final.csv', index=False)
rf_search_df.to_csv(BASE_DIR / 'hiperparametros_random_forest_final.csv', index=False)
report_mlp_df.to_csv(BASE_DIR / 'classification_report_mlp_final.csv', index=False)
report_rf_df.to_csv(BASE_DIR / 'classification_report_random_forest_final.csv', index=False)
importance_df.to_csv(BASE_DIR / 'importancia_variables_random_forest_final.csv', index=False)

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(cm_mlp, annot=True, fmt='.2f', cmap='Blues', xticklabels=labels, yticklabels=labels, ax=ax)
ax.set_title('Matriz de confusión normalizada - MLP principal')
ax.set_xlabel('Clase predicha')
ax.set_ylabel('Clase real')
plt.tight_layout()
plt.savefig(BASE_DIR / 'confusion_matrix_mlp_final.png', dpi=200)
plt.close(fig)

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(cm_rf, annot=True, fmt='.2f', cmap='Greens', xticklabels=labels, yticklabels=labels, ax=ax)
ax.set_title('Matriz de confusión normalizada - Random Forest final')
ax.set_xlabel('Clase predicha')
ax.set_ylabel('Clase real')
plt.tight_layout()
plt.savefig(BASE_DIR / 'confusion_matrix_random_forest_final.png', dpi=200)
plt.close(fig)

print('Archivos actualizados en:', BASE_DIR)
print('Gráficos exportados:')
for file_name in [
    'grafica_distribucion_clases.png',
    'graficas_distribucion_variables_continuas.png',
    'matriz_correlacion.png',
    'boxplots_variables_por_clase.png',
    'arquitectura_mlp_principal.png',
    'curvas_entrenamiento_mlp.png',
    'comparacion_hiperparametros_mlp.png',
    'comparacion_hiperparametros_random_forest.png',
    'matrices_confusion_comparadas.png',
    'comparacion_metricas_modelos.png',
    'importancia_variables_random_forest.png',
    'confusion_matrix_mlp_final.png',
    'confusion_matrix_random_forest_final.png',
]:
    print(BASE_DIR / file_name)

## 13. Observaciones para Overleaf

- Incluir una figura con la **distribución de clases**.
- Incluir una figura con la **matriz de correlación**.
- Incluir una tabla con la **búsqueda de hiperparámetros del MLP**.
- Incluir una tabla comparativa entre **MLP principal** y **Random Forest**.
- Incluir las **matrices de confusión normalizadas** de ambos modelos.
- Incluir una figura con la **importancia de variables** del Random Forest.
- En la discusión, destacar que el `Random Forest` superó al MLP en este problema tabular.
- Justificar el modelo adicional con literatura relacionada de predicción de rendimiento académico y deserción.